In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Tuple
import os

os.makedirs('result', exist_ok=True)

In [ ]:
# =============================================================================
# Cell 1: 데이터 로딩 및 전처리
# =============================================================================

DATA_PATH = r'../crawling/data/kospi200_option_B0164800_20260406_0921.csv'
start_row = 30
N_STEPS = 200

df_raw = pd.read_csv(DATA_PATH, encoding='utf-8-sig')

# NaN 보간 (index 15-19 등 완전 NaN 행)
numeric_cols = df_raw.select_dtypes(include='number').columns
df_raw[numeric_cols] = df_raw[numeric_cols].interpolate(method='linear')
df_raw = df_raw.ffill().bfill()

# 경계 체크 (N_STEPS + 1행 필요: step 0 초기화 + N_STEPS 거래)
assert start_row + N_STEPS + 1 <= len(df_raw), \
    f"start_row({start_row}) + N_STEPS+1({N_STEPS+1}) > 데이터 길이({len(df_raw)})"

df = df_raw.iloc[start_row : start_row + N_STEPS + 1].reset_index(drop=True)

# 경로 추출
stock_prices = df['지수현재가'].values.astype(float)
option_prices = ((df['옵션매도호가1'] + df['옵션매수호가1']) / 2).values.astype(float)
deltas = df['옵션델타'].values.astype(float)

print(f"데이터 범위: {df_raw['시간'].iloc[start_row]} ~ {df_raw['시간'].iloc[start_row + N_STEPS]}")
print(f"지수: {stock_prices[0]:.2f} -> {stock_prices[-1]:.2f}")
print(f"옵션 mid (bid1+ask1)/2: {option_prices[0]:.2f} -> {option_prices[-1]:.2f}")
print(f"델타: {deltas[0]:.4f} -> {deltas[-1]:.4f}")
print(f"배열 길이: {len(stock_prices)} (N_STEPS+1={N_STEPS+1})")

In [ ]:
# =============================================================================
# Cell 2: OptimizeConfig (train과 동일한 파라미터, 경로 생성 파라미터 제외)
# =============================================================================

@dataclass
class OptimizeConfig:
    # A-S model parameters
    gamma: float = 0.1
    A: float = 140.0
    k: float = 1.5
    q0: int = 0
    dt: float = 0.005
    n_steps: int = 200

    # Develop extensions
    n_levels: int = 3
    tick_size: float = 0.05
    buckets: List[Tuple[float, float]] = field(default_factory=lambda: [
        (0.02, 1.0), (0.05, 1.5), (0.10, 2.5), (float('inf'), 3.5)
    ])

    # GARCH
    garch_vl: float = 4.0

    # Reward
    lambda_inv: float = 0.5

config = OptimizeConfig()

In [ ]:
# =============================================================================
# Cell 3: Core functions (train.ipynb과 동일)
# =============================================================================

def tick_floor(price, tick_size):
    return np.floor(price / tick_size) * tick_size

def tick_ceil(price, tick_size):
    return np.ceil(price / tick_size) * tick_size

def get_fill_multiplier(price_change_pct, config):
    abs_change = abs(price_change_pct)
    for threshold, multiplier in config.buckets:
        if abs_change < threshold:
            return multiplier
    return config.buckets[-1][1]

def compute_fill_probability(delta, stock_prev, stock_current, is_bid, config):
    if delta <= 0:
        return 1.0
    base_intensity = config.A * np.exp(-config.k * delta)
    base_prob = min(base_intensity * config.dt, 1.0)
    stock_change_pct = (stock_current - stock_prev) / stock_prev * 100
    multiplier = get_fill_multiplier(stock_change_pct, config)
    if stock_change_pct > 0:
        if not is_bid:
            return min(base_prob * multiplier, 1.0)
        else:
            return max(base_prob / multiplier, 0.0)
    elif stock_change_pct < 0:
        if is_bid:
            return min(base_prob * multiplier, 1.0)
        else:
            return max(base_prob / multiplier, 0.0)
    return base_prob

def compute_rp(s, net_delta, sigma_new_sq, config):
    return s - net_delta * config.gamma * sigma_new_sq

def compute_spread(sigma_new_sq, config):
    return config.gamma * sigma_new_sq + (2 / config.gamma) * np.log(1 + config.gamma / config.k)

def compute_quotes(s, net_delta, sigma_new_sq, config):
    rp = compute_rp(s, net_delta, sigma_new_sq, config)
    spread = compute_spread(sigma_new_sq, config)
    bid = tick_floor(rp - spread / 2, config.tick_size)
    ask = tick_ceil(rp + spread / 2, config.tick_size)
    return bid, ask

In [ ]:
# =============================================================================
# Cell 4: MMEnvironment (실제 데이터 버전)
# =============================================================================
# train.ipynb의 MMEnvironment에서 경로 생성을 제거하고,
# 외부에서 stock_prices, option_prices, deltas 배열을 주입받는 구조

class MMEnvironment:

    def __init__(self, config, stock_prices, option_prices, deltas):
        self.config = config
        self.stock_prices = stock_prices
        self.option_prices = option_prices
        self.deltas = deltas

    def reset(self, seed=None):
        if seed is not None:
            np.random.seed(seed)

        c = self.config
        self.sigma_sq_path = np.zeros(c.n_steps + 1)
        self.sigma_sq_path[0] = c.garch_vl

        self.inventory = c.q0
        self.cash = 0.0
        self.t = 0
        self.max_abs_inventory = 0
        self.prev_return = 0.0

        # step 0: 호가 설정 (1-step lag)
        s0 = self.option_prices[0]
        net_delta_0 = self.inventory * self.deltas[0]
        bid0, ask0 = compute_quotes(s0, net_delta_0, self.sigma_sq_path[0], c)
        self.bid_levels_prev = [bid0 - i * c.tick_size for i in range(c.n_levels)]
        self.ask_levels_prev = [ask0 + i * c.tick_size for i in range(c.n_levels)]

        return self._get_state()

    def _get_state(self):
        return {
            'net_delta': self.inventory * self.deltas[self.t],
            'sigma_sq_garch': self.sigma_sq_path[self.t],
            'inventory': self.inventory,
            'step': self.t,
            'prev_return': self.prev_return,
        }

    def step_continuous(self, alpha_t, beta_t):
        c = self.config
        self.t += 1
        if self.t >= c.n_steps:
            return self._get_state(), 0.0, True

        s = self.option_prices[self.t]
        s_stock = self.stock_prices[self.t]
        s_stock_prev = self.stock_prices[self.t - 1]
        delta_t = self.deltas[self.t]

        # 체결 판정 (t-1 호가 vs t시점 mid)
        bid_fills = [False] * c.n_levels
        ask_fills = [False] * c.n_levels
        for i in range(c.n_levels):
            delta_bid = s - self.bid_levels_prev[i]
            delta_ask = self.ask_levels_prev[i] - s
            prob_bid = compute_fill_probability(delta_bid, s_stock_prev, s_stock, True, c)
            prob_ask = compute_fill_probability(delta_ask, s_stock_prev, s_stock, False, c)
            bid_fills[i] = np.random.random() < prob_bid
            ask_fills[i] = np.random.random() < prob_ask

        # sweep 보정
        for i in range(c.n_levels - 1, 0, -1):
            if bid_fills[i]:
                bid_fills[i - 1] = True
            if ask_fills[i]:
                ask_fills[i - 1] = True

        # 체결 실행
        total_bid_cost = 0.0
        total_ask_revenue = 0.0
        bid_fill_count = 0
        ask_fill_count = 0
        for i in range(c.n_levels):
            if bid_fills[i]:
                delta_bid = s - self.bid_levels_prev[i]
                fill_price = s if delta_bid <= 0 else self.bid_levels_prev[i]
                total_bid_cost += fill_price
                bid_fill_count += 1
            if ask_fills[i]:
                delta_ask = self.ask_levels_prev[i] - s
                fill_price = s if delta_ask <= 0 else self.ask_levels_prev[i]
                total_ask_revenue += fill_price
                ask_fill_count += 1

        self.inventory += bid_fill_count - ask_fill_count
        self.cash += total_ask_revenue - total_bid_cost
        self.max_abs_inventory = max(self.max_abs_inventory, abs(self.inventory))

        # GARCH 업데이트 (alpha, beta를 policy에서 받음)
        omega = (1 - alpha_t - beta_t) * c.garch_vl
        r_t = s_stock - s_stock_prev  # 지수 절대 변화량
        self.prev_return = r_t
        self.sigma_sq_path[self.t] = omega + alpha_t * r_t**2 + beta_t * self.sigma_sq_path[self.t - 1]

        # 새 호가 설정
        sigma_new_sq = self.sigma_sq_path[self.t]
        net_delta_after = self.inventory * delta_t
        bid_new, ask_new = compute_quotes(s, net_delta_after, sigma_new_sq, c)
        self.bid_levels_prev = [bid_new - i * c.tick_size for i in range(c.n_levels)]
        self.ask_levels_prev = [ask_new + i * c.tick_size for i in range(c.n_levels)]

        done = (self.t >= c.n_steps - 1)
        return self._get_state(), sigma_new_sq, done

    def get_final_reward(self):
        c = self.config
        final_pnl = self.cash + self.inventory * self.option_prices[min(self.t, c.n_steps)]
        penalty = c.lambda_inv * self.max_abs_inventory
        return final_pnl - penalty, final_pnl, self.max_abs_inventory

In [ ]:
# =============================================================================
# Cell 5: MMPolicy (train.ipynb과 동일)
# =============================================================================

ALPHA_MIN = 0.03
ALPHA_MAX = 0.15
BETA_SUM = 0.97

def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))


class MMPolicy:

    def __init__(self, theta):
        self.theta = theta

    def get_alpha_beta(self, state):
        score = (abs(state['net_delta']) * self.theta[0]
                 + abs(state['prev_return']) * self.theta[1])
        sig = sigmoid((score - self.theta[2]) / max(self.theta[3], 0.1))
        alpha_t = ALPHA_MIN + (ALPHA_MAX - ALPHA_MIN) * sig
        beta_t = BETA_SUM - alpha_t
        return alpha_t, beta_t

    def run_episode(self, env, seed=None):
        state = env.reset(seed=seed)
        alpha_history = []
        sigma_history = []
        done = False
        while not done:
            alpha_t, beta_t = self.get_alpha_beta(state)
            alpha_history.append(alpha_t)
            state, sigma_sq, done = env.step_continuous(alpha_t, beta_t)
            sigma_history.append(sigma_sq)
        reward, pnl, max_inv = env.get_final_reward()
        return reward, pnl, max_inv, alpha_history, sigma_history

In [ ]:
# =============================================================================
# Cell 6: Baseline 단일 시뮬레이션 시각화
# =============================================================================

env = MMEnvironment(config, stock_prices, option_prices, deltas)
baseline_policy = MMPolicy(np.array([1.0, 1.0, 5.0, 2.0]))
state = env.reset(seed=42)

inv_rec = [0]
nd_rec = [0.0]
pnl_rec = [0.0]
bid_rec = [env.bid_levels_prev[0]]
ask_rec = [env.ask_levels_prev[0]]
sigma_rec = [config.garch_vl]
alpha_rec = []

done = False
while not done:
    alpha_t, beta_t = baseline_policy.get_alpha_beta(state)
    alpha_rec.append(alpha_t)
    state, sigma_sq, done = env.step_continuous(alpha_t, beta_t)
    inv_rec.append(env.inventory)
    nd_rec.append(state['net_delta'])
    pnl_rec.append(env.cash + env.inventory * env.option_prices[env.t])
    bid_rec.append(env.bid_levels_prev[0])
    ask_rec.append(env.ask_levels_prev[0])
    sigma_rec.append(sigma_sq if sigma_sq else sigma_rec[-1])

reward_b, pnl_b, max_inv_b = env.get_final_reward()
n = len(alpha_rec)
steps = np.arange(n + 1)
steps_alpha = np.arange(1, n + 1)

fig, axes = plt.subplots(6, 1, figsize=(12, 20))

axes[0].plot(steps, stock_prices[:n+1], 'b-', linewidth=1)
axes[0].set_ylabel('KOSPI200 Index')
axes[0].set_title(f'Optimize: Real Data (start_row={start_row})')
axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, option_prices[:n+1], 'b-', label='Option Mid', linewidth=1)
axes[1].plot(steps, bid_rec[:n+1], 'g--', label='Bid', alpha=0.7, linewidth=0.5)
axes[1].plot(steps, ask_rec[:n+1], 'r--', label='Ask', alpha=0.7, linewidth=0.5)
axes[1].set_ylabel('Option Price')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(steps, nd_rec, 'purple', linewidth=1)
axes[2].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[2].set_ylabel('Net Delta')
axes[2].grid(True, alpha=0.3)

axes[3].plot(steps_alpha, alpha_rec, 'darkorange', linewidth=1)
axes[3].axhline(y=ALPHA_MIN, color='green', linestyle='--', alpha=0.5, label=f'alpha_min={ALPHA_MIN}')
axes[3].axhline(y=ALPHA_MAX, color='red', linestyle='--', alpha=0.5, label=f'alpha_max={ALPHA_MAX}')
axes[3].set_ylabel('alpha_t (GARCH)')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

axes[4].plot(steps_alpha, sigma_rec[1:n+1], 'red', linewidth=1)
axes[4].axhline(y=config.garch_vl, color='black', linestyle='--', alpha=0.3, label=f'V_L={config.garch_vl}')
axes[4].set_ylabel('sigma_new^2')
axes[4].legend()
axes[4].grid(True, alpha=0.3)

axes[5].plot(steps, pnl_rec, 'orange', linewidth=1)
axes[5].set_xlabel('Step (1 step = 1 min)')
axes[5].set_ylabel('P&L')
axes[5].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('result/optimize_baseline.png', dpi=150)
plt.show()

print(f'Final P&L: {pnl_b:.2f}, Reward: {reward_b:.2f}')
print(f'Max |inventory|: {max_inv_b}, Final inventory: {env.inventory}')
print(f'Alpha range: {min(alpha_rec):.4f} ~ {max(alpha_rec):.4f}')
print(f'Sigma^2 range: {min(sigma_rec[1:]):.4f} ~ {max(sigma_rec[1:]):.4f}')

In [ ]:
# =============================================================================
# Cell 7: Constant baseline vs Learned policy 비교
# =============================================================================

# train.ipynb에서 학습된 best_theta를 여기에 입력
learned_theta = np.array([1.0, 1.0, 5.0, 2.0])  # TODO: train 결과로 교체

# 상수 baseline: center=999 -> sigmoid ~ 0 -> alpha ~ 0.03 항상
constant_policy = MMPolicy(np.array([1.0, 1.0, 999.0, 1.0]))
learned_policy = MMPolicy(learned_theta)

n_eval = 200

const_results = {'reward': [], 'pnl': [], 'max_inv': []}
learn_results = {'reward': [], 'pnl': [], 'max_inv': []}

for i in range(n_eval):
    env_c = MMEnvironment(config, stock_prices, option_prices, deltas)
    r, p, m, _, _ = constant_policy.run_episode(env_c, seed=i)
    const_results['reward'].append(r)
    const_results['pnl'].append(p)
    const_results['max_inv'].append(m)

    env_l = MMEnvironment(config, stock_prices, option_prices, deltas)
    r, p, m, _, _ = learned_policy.run_episode(env_l, seed=i)
    learn_results['reward'].append(r)
    learn_results['pnl'].append(p)
    learn_results['max_inv'].append(m)

print(f"{'':12} {'Reward':>10} {'P&L':>10} {'P&L Std':>10} {'Max|Inv|':>10}")
print('-' * 55)
for name, res in [('Constant', const_results), ('Learned', learn_results)]:
    print(f"{name:12} {np.mean(res['reward']):10.2f} {np.mean(res['pnl']):10.2f} "
          f"{np.std(res['pnl']):10.2f} {np.mean(res['max_inv']):10.1f}")

print(f'\nLearned theta: {learned_theta}')

# P&L 분포 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(const_results['pnl'], bins=30, alpha=0.7, label='Constant', color='blue')
axes[0].hist(learn_results['pnl'], bins=30, alpha=0.7, label='Learned', color='green')
axes[0].set_xlabel('Final P&L')
axes[0].set_ylabel('Frequency')
axes[0].set_title('P&L Distribution: Constant vs Learned')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(const_results['max_inv'], bins=30, alpha=0.7, label='Constant', color='blue')
axes[1].hist(learn_results['max_inv'], bins=30, alpha=0.7, label='Learned', color='green')
axes[1].set_xlabel('Max |Inventory|')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Max Inventory Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('result/optimize_comparison.png', dpi=150)
plt.show()